In [0]:
bronze_table = "banking_project.bronze.bronze_transactions"
silver_table = "banking_project.silver.silver_transactions"

df_bronze = spark.table(bronze_table)

print("Bronze record count:", df_bronze.count())
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, to_date

df_silver = (
    df_bronze
    .withColumn(
        "transaction_amount",
        col("transaction_amount").cast("decimal(18,2)")
    )
    .withColumn(
        "transaction_date",
        to_date(col("transaction_date"))
    )
)

df_silver.select(
    "transaction_id",
    "transaction_amount",
    "transaction_date"
).show(5)

df_silver.select(
    "transaction_amount",
    "transaction_date"
).printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

df_silver.select(
    spark_sum(
        when(col("transaction_id").isNull(), 1).otherwise(0)
    ).alias("missing_transaction_ids"),

    spark_sum(
        when(col("transaction_amount").isNull(), 1).otherwise(0)
    ).alias("invalid_or_missing_amounts"),

    spark_sum(
        when(col("transaction_date").isNull(), 1).otherwise(0)
    ).alias("invalid_or_missing_dates")
).show()

In [0]:
df_duplicates = (
    df_silver
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
)

print("Duplicate transaction IDs:", df_duplicates.count())

df_duplicates.show(10)

In [0]:
from pyspark.sql.functions import get_json_object, col

df_silver = df_silver.withColumn(
    "device_type",
    get_json_object(col("_rescued_data"), "$.device_type")
)

df_silver.select(
    "transaction_id",
    "device_type"
).filter(
    col("device_type").isNotNull()
).show(15)

In [0]:
from pyspark.sql import functions as F

df_silver.select(
    F.sum(
        F.when(F.col("transaction_amount") <= 0, 1).otherwise(0)
    ).alias("non_positive_amounts"),

    F.sum(
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_customer_ids"),

    F.sum(
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_or_empty_transaction_ids")
).show()

In [0]:
from pyspark.sql import functions as F

# Read the existing Bronze Delta table
df_bronze = spark.table(
    "banking_project.bronze.bronze_transactions"
)

# Recreate Silver transformations
df_silver = (
    df_bronze
    .withColumn(
        "transaction_amount",
        F.col("transaction_amount").cast("decimal(18,2)")
    )
    .withColumn(
        "transaction_date",
        F.to_date("transaction_date")
    )
    .withColumn(
        "device_type",
        F.get_json_object(
            F.col("_rescued_data"),
            "$.device_type"
        )
    )
)

print("Silver DataFrame records:", df_silver.count())

In [0]:

from pyspark.sql import functions as F

# Select 3 sample transactions
sample_rows = (
    df_silver
    .orderBy("transaction_id")
    .limit(3)
    .collect()
)

# Create invalid test records
df_invalid_test = spark.createDataFrame(
    sample_rows,
    schema=df_silver.schema
)

# Give each record a deliberate data quality problem
df_invalid_test = (
    df_invalid_test
    .withColumn(
        "transaction_id",
        F.when(F.col("transaction_id") == sample_rows[0]["transaction_id"], "TEST001")
         .when(F.col("transaction_id") == sample_rows[1]["transaction_id"], "TEST002")
         .otherwise("")
    )
    .withColumn(
        "transaction_amount",
        F.when(F.col("transaction_id") == "TEST001", F.lit(-500).cast("decimal(18,2)"))
         .otherwise(F.col("transaction_amount"))
    )
    .withColumn(
        "customer_id",
        F.when(F.col("transaction_id") == "TEST002", F.lit(None).cast("string"))
         .otherwise(F.col("customer_id"))
    )
)

df_invalid_test.select(
    "transaction_id",
    "transaction_amount",
    "customer_id"
).show(truncate=False)


In [0]:

from pyspark.sql import functions as F

# Add rejection reasons to invalid records
df_validated = df_invalid_test.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("transaction_amount").isNull() |
            (F.col("transaction_amount") <= 0),
            "INVALID_AMOUNT"
        ),
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        ),
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "MISSING_TRANSACTION_ID"
        )
    )
)

# Separate invalid records
df_quarantine = df_validated.filter(
    F.col("rejection_reason") != ""
)

# Separate valid records
df_valid = df_validated.filter(
    F.col("rejection_reason") == ""
)

print("Total test records:", df_validated.count())
print("Quarantined records:", df_quarantine.count())
print("Valid records:", df_valid.count())

df_quarantine.select(
    "transaction_id",
    "transaction_amount",
    "customer_id",
    "rejection_reason"
).show(truncate=False)


In [0]:
from pyspark.sql import functions as F

df_validated_all = df_silver.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("transaction_amount").isNull() |
            (F.col("transaction_amount") <= 0),
            "INVALID_AMOUNT"
        ),
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        ),
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "MISSING_TRANSACTION_ID"
        ),
        F.when(
            F.col("transaction_date").isNull(),
            "INVALID_TRANSACTION_DATE"
        )
    )
)

df_valid_all = df_validated_all.filter(
    F.col("rejection_reason") == ""
)

df_quarantine_all = df_validated_all.filter(
    F.col("rejection_reason") != ""
)

print("Bronze records:", df_bronze.count())
print("Valid records:", df_valid_all.count())
print("Quarantined records:", df_quarantine_all.count())

In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_silver
    .withColumn(
        "account_balance",
        F.expr("try_cast(account_balance AS DECIMAL(18,2))")
    )
    .withColumn(
        "emi_amount",
        F.expr("try_cast(emi_amount AS DECIMAL(18,2))")
    )
    .withColumn(
        "credit_score",
        F.expr("try_cast(credit_score AS INT)")
    )
    .withColumn(
        "transaction_hour",
        F.expr("try_cast(transaction_hour AS INT)")
    )
)

df_silver.select(
    "account_balance",
    "emi_amount",
    "credit_score",
    "transaction_hour"
).printSchema()

df_silver.select(
    "account_balance",
    "emi_amount",
    "credit_score",
    "transaction_hour"
).show(5)

In [0]:
from pyspark.sql import functions as F

df_silver.select(
    F.sum(
        F.when(F.col("account_balance").isNull(), 1).otherwise(0)
    ).alias("invalid_account_balance"),

    F.sum(
        F.when(F.col("emi_amount").isNull(), 1).otherwise(0)
    ).alias("invalid_emi_amount"),

    F.sum(
        F.when(
            F.col("credit_score").isNotNull() &
            ~F.col("credit_score").between(300, 900),
            1
        ).otherwise(0)
    ).alias("invalid_credit_score"),

    F.sum(
        F.when(
            F.col("transaction_hour").isNull() |
            ~F.col("transaction_hour").between(0, 23),
            1
        ).otherwise(0)
    ).alias("invalid_transaction_hour")
).show()

In [0]:
# Count records before deduplication
before_count = df_silver.count()

# Keep one record per transaction ID
df_silver_dedup = df_silver.dropDuplicates(
    ["transaction_id"]
)

# Count records after deduplication
after_count = df_silver_dedup.count()

print("Before deduplication:", before_count)
print("After deduplication:", after_count)
print("Duplicates removed:", before_count - after_count)

In [0]:
from pyspark.sql import functions as F

df_final_checked = df_silver_dedup.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",

        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "MISSING_TRANSACTION_ID"
        ),

        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        ),

        F.when(
            F.col("transaction_amount").isNull() |
            (F.col("transaction_amount") <= 0),
            "INVALID_AMOUNT"
        ),

        F.when(
            F.col("transaction_date").isNull(),
            "INVALID_DATE"
        ),

        F.when(
            F.col("account_balance").isNull(),
            "INVALID_ACCOUNT_BALANCE"
        ),

        F.when(
            F.col("emi_amount").isNull(),
            "INVALID_EMI_AMOUNT"
        ),

        F.when(
            F.col("credit_score").isNotNull() &
            ~F.col("credit_score").between(300, 900),
            "INVALID_CREDIT_SCORE"
        ),

        F.when(
            F.col("transaction_hour").isNull() |
            ~F.col("transaction_hour").between(0, 23),
            "INVALID_TRANSACTION_HOUR"
        )
    )
)

# Separate accepted and rejected transactions
df_final_valid = df_final_checked.filter(
    F.col("rejection_reason") == ""
).drop("rejection_reason")

df_final_quarantine = df_final_checked.filter(
    F.col("rejection_reason") != ""
)

print("Total records:", df_final_checked.count())
print("Valid records:", df_final_valid.count())
print("Quarantined records:", df_final_quarantine.count())

In [0]:
silver_table = "banking_project.silver.silver_transactions"

if not spark.catalog.tableExists(silver_table):

    (
        df_final_valid.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    print("Silver Delta table created successfully!")

else:
    print("Silver table already exists. No records written.")

In [0]:
%sql
SELECT COUNT(*)
FROM banking_project.silver.silver_transactions;